In [1]:
import pickle
import sys
import numpy as np
import pandas as pd
from pathlib import Path
from tqdm.auto import tqdm

sys.path.append("..")
from utils.feature_extraction import extract_rhythm_features


C:\Users\ankit\miniconda3\envs\ml\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
DATA_DIR = Path("../data/clean")
OUTPUT_DIR = Path("../data/extracted")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

GENRE_FILES = [
    "Bauls_clean.pkl",
    "Bhatiali_clean.pkl",
    "Gidha_clean.pkl",
    "Kajri_clean.pkl",
    "Maand_clean.pkl",
    "Sohar_clean.pkl",
    "Sufi_clean.pkl",
    "Uttarakhandi_clean.pkl",
]


In [3]:
def load_pickle(path: Path) -> dict:
    with open(path, "rb") as f:
        data = pickle.load(f)
    return data


In [ ]:
all_rows = []

for filename in tqdm(GENRE_FILES, desc="Genre files"):
    file_path = DATA_DIR / filename
    data = load_pickle(file_path)

    mel_specs = data["mel_spec"]
    n_samples = mel_specs.shape[0]

    genre = data.get("genre", None)
    state = data.get("state", None)
    artist = data.get("artist", None)
    gender = data.get("gender", None)
    song = data.get("song", None)
    source = data.get("source", None)

    def get_field(field, i):
        if field is None:
            return None
        if isinstance(field, (list, np.ndarray)):
            return field[i]
        return field

    for i in tqdm(range(n_samples), desc=filename, leave=False):
        mel_spec = mel_specs[i]
        features = extract_rhythm_features(mel_spec)

        row = {
            "genre": get_field(genre, i),
            "state": get_field(state, i),
            "artist": get_field(artist, i),
            "gender": get_field(gender, i),
            "song": get_field(song, i),
            "source": get_field(source, i),
            "source_file": filename,
        }
        row.update(features)
        all_rows.append(row)


In [5]:
rhythm_df = pd.DataFrame(all_rows)
rhythm_df.shape


(109089, 41)

In [6]:
rhythm_df.head()


,genre,state,artist,gender,song,source,source_file,energy_mean,energy_std,energy_max,...,rhythm_hist_std,tempogram_dominant_lag,tempogram_stability,tempogram_mean_energy,tempogram_var,onset_mean,onset_std,onset_max,onset_peak_count,onset_density
0,Bauls,West Bengal,Babul Supriyo,Male,Sundori Kamala,https://gaana.com/song/sundori-kamala-1,Bauls_clean.pkl,-6664.115234,1794.571289,-4018.647217,...,0.080684,1.0,1.000000e+10,0.015152,0.087408,296.212219,549.835083,3057.824219,13.0,0.100775
1,Bauls,West Bengal,Babul Supriyo,Male,Sundori Kamala,https://gaana.com/song/sundori-kamala-1,Bauls_clean.pkl,-6034.185547,988.613953,-4096.845703,...,0.019924,1.0,1.000000e+10,0.015152,0.080058,329.697266,559.271729,2703.819336,14.0,0.108527
2,Bauls,West Bengal,Babul Supriyo,Male,Sundori Kamala,https://gaana.com/song/sundori-kamala-1,Bauls_clean.pkl,-5896.586426,977.628967,-3886.344971,...,0.019915,1.0,1.000000e+10,0.015152,0.077731,328.702515,526.732056,2139.476562,16.0,0.124031
3,Bauls,West Bengal,Babul Supriyo,Male,Sundori Kamala,https://gaana.com/song/sundori-kamala-1,Bauls_clean.pkl,-5900.519043,1017.549377,-3920.866699,...,0.021060,1.0,1.000000e+10,0.015152,0.074588,331.991638,526.898865,2011.021484,16.0,0.124031
4,Bauls,West Bengal,Babul Supriyo,Male,Sundori Kamala,https://gaana.com/song/sundori-kamala-1,Bauls_clean.pkl,-5906.877930,991.570679,-3973.443115,...,0.022325,1.0,1.000000e+10,0.015152,0.074591,341.991150,556.537292,2182.362305,15.0,0.116279


In [7]:
parquet_path = OUTPUT_DIR / "rhythm_features.parquet"
pickle_path = OUTPUT_DIR / "rhythm_features.pkl"

rhythm_df.to_parquet(parquet_path, index=False)

with open(pickle_path, "wb") as f:
    pickle.dump(rhythm_df, f)

parquet_path, pickle_path


(WindowsPath('../data/extracted/rhythm_features.parquet'),
 WindowsPath('../data/extracted/rhythm_features.pkl'))